# Pipeline de preprocessing : Adult Income (UCI)

La pipeline est **ajustée sur le train uniquement** (médianes, modes, échelles), puis appliquée au test pour éviter la fuite de données.

## 0. Préparation de l'environnement

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from adult_income import (
    DataLoader,
    PreprocessingPipeline,
    NUMERICAL_FEATURES,
    CATEGORICAL_FEATURES,
    TARGET,
    POSITIVE_CLASS,
    NEGATIVE_CLASS,
)
from adult_income.data.schemas import (
    NORMAL_NUMERICAL_FEATURES,
    LOG_NUMERICAL_FEATURES,
    BINARY_NUMERICAL_FEATURES,
    MODE_CATEGORICAL_FEATURES,
    UNKNOWN_ONEHOT_FEATURES,
    GROUPED_CATEGORICAL_FEATURES,
    RARE_CATEGORY_MIN_FREQUENCY,
    FEATURES_TO_DROP,
)

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

print("Imports OK")


## 1. Chargement et découpage train / test

In [ ]:
loader = DataLoader()  # split officiel : adult.data = train, adult.test = test
X_train, X_test, y_train, y_test = loader.load_and_split()

print(f"Train : {X_train.shape[0]:,} obs  -  Test : {X_test.shape[0]:,} obs")
print(f"Features brutes : {X_train.shape[1]}")
X_train.head()


Etant textuelle, `income` (`>50K` / `<=50K`) est encodée en 0/1 via le `LabelEncoder` interne de la pipeline.

Concernant le paramètre capital_treatment de la pipeline de preproc, la valeur "hurdle" peut être choisie et concerne le traitement des deux variables `capital-*`

In [ ]:
pipe = PreprocessingPipeline(scaling=True)

y_train_enc = pipe.fit_transform_target(y_train)
y_test_enc = pipe.transform_target(y_test)

print("Classes (ordre LabelEncoder) :", list(pipe.classes_))
print("Répartition train encodée   :", np.bincount(y_train_enc))

## 2. La pipeline de preprocessing : vue d'ensemble

`build()` retourne un `ColumnTransformer` prêt à être ajusté. On affiche son
diagramme : chaque branche cible un sous-ensemble de colonnes (les features
listées dans `FEATURES_TO_DROP` sont ignorées via `remainder="drop"`).

In [ ]:
print("Numériques normales (StandardScaler) :", NORMAL_NUMERICAL_FEATURES)
print("Numériques log1p (RobustScaler)       :", LOG_NUMERICAL_FEATURES)
print("Indicateurs binaires has_*            :", BINARY_NUMERICAL_FEATURES)
print("Catégorielles one-hot (mode)          :", MODE_CATEGORICAL_FEATURES)
print("Catégorielles one-hot + 'Unknown'     :", UNKNOWN_ONEHOT_FEATURES)
print(f"Catégorielles regroupées (<{RARE_CATEGORY_MIN_FREQUENCY})        :", GROUPED_CATEGORICAL_FEATURES)
print("Features ignorées (drop)              :", FEATURES_TO_DROP)

pre = pipe.build()
pre

## 3. Application de la pipeline (`fit_transform` sur le train)

In [ ]:
X_train_t = pre.fit_transform(X_train, y_train_enc)
feat_names = pre.get_feature_names_out()
X_train_t = pd.DataFrame(X_train_t, columns=feat_names, index=X_train.index)

print(f"Avant : {X_train.shape[1]} colonnes  ->  Apres : {X_train_t.shape[1]} colonnes")
X_train_t.head()

## 4. Vérification des effets du prétraitement

Pour chaque décision du **bilan AED**, on contrôle juste que la pipeline l'applique correctement, et on regarde les effets notamment pour le log1p

### 4.1 Valeurs manquantes & `?`

Les `?` du dataset Adult sont convertis en `NaN` dès le chargement.
La pipeline garantit qu'il ne reste aucune valeur manquante en sortie, mais avec une
stratégie par variable plutôt qu'une imputation uniforme par le mode :

- **`workclass` / `occupation`** → catégorie **`Unknown`** 
- **`native-country`** → imputation par le **mode** 


In [ ]:
# Les ? sont deja convertis en NaN au chargement -> on compte les NaN
raw_missing = X_train[CATEGORICAL_FEATURES].isna().sum()
print("Valeurs manquantes (NaN) dans le train brut :")
print(raw_missing[raw_missing > 0].to_string())

n_nan = int(np.isnan(X_train_t.values).sum())
print(f"\nNaN dans la matrice transformee : {n_nan}")

### 4.2 Mise à l'échelle des variables numériques

Le `StandardScaler` recentre/réduit certaines features numériques (indispensable pour les
modèles sensibles à l'échelle : régression logistique, SVM, KNN). La forme des
distributions ne change pas par rapport à l'EDA. On vérifie simplement qu'en sortie la
**moyenne ≈ 0** et l'**écart-type ≈ 1**.

In [ ]:
scaled = X_train_t[[f"num__{c}" for c in NORMAL_NUMERICAL_FEATURES]].copy()
scaled.columns = NORMAL_NUMERICAL_FEATURES
scaled.agg(["mean", "std"]).round(3)

### 4.3 Variables `capital-*` : log1p + indicateurs binaires (hurdle)

`capital-gain` et `capital-loss` sont très
asymétriques et ont beaucoup de 0. La proposition AED les décompose façon « hurdle » :

- **`capital-gain`** → `log1p` + `RobustScaler` : on comprime la queue lourde (valeurs très extrêmes)
  tout en conservant la magnitude (dispersion des valeurs et écarts entre-elles) (un gros gain semble très prédictif de `>50K`).
- **`capital-loss`** → pas de log : sa magnitude est concentrée et peu
  informative, un simple indicateur binaire suffit.
- **Indicateurs `has_capital-gain` / `has_capital-loss`** :
  captent le signal zéro / non-zéro, potentiellement plus utile pour `has_capital-loss` (car pas d'extrêmes) 

Cette proposition n'est pas un choix final et peut être confirmée ou infirmée lorsque nous ferons les ablations dans la modélisation

In [ ]:
fig, prop = pipe.plot_capital_treatment(X_train, X_train_t)
print("Part d'individus avec un capital non nul :")
print(prop.to_string())
fig;

### 4.4 Encodage one-hot des catégorielles

Deux branches one-hot :

- **`cat`** (marital-status, relationship, race, sex) : imputation par le mode
  (sans effet ici, car ces colonnes n'ont aucun manquant) puis one-hot. Toutes les
  modalités sont conservées.
- **`cat_unknown`** (`workclass`, `occupation`) : les valeurs manquantes sont informatives différement du mode,
  donc les `?` deviennent une catégorie `Unknown` à part entière (`workclass_Unknown`,
  `occupation_Unknown`) au lieu d'être imputés par le mode.
  `occupation` (15 modalités) reste gérable en one-hot.

In [ ]:
onehot_bases = MODE_CATEGORICAL_FEATURES + UNKNOWN_ONEHOT_FEATURES
onehot_cols = [c for c in feat_names if c.startswith("cat__") or c.startswith("cat_unknown__")]
print(f"{len(onehot_bases)} variables one-hot  ->  {len(onehot_cols)} colonnes\n")
for base in onehot_bases:
    n = sum(f"__{base}_" in c for c in feat_names)
    print(f"  {base:<16} -> {n:>3} modalites")

unknown_col = "cat_unknown__workclass_Unknown"
print(f"\n{unknown_col} : {int(X_train_t[unknown_col].sum())} individus "
      f"(= manquants workclass conservés, pas imputés)")


### 4.5 `native-country` : regroupement des modalités rares

**`native-country`** a 41 modalités (~90 % US), et ce qui s'apparente à du bruit sur les faibles
effectifs. Les valeurs manquantes ne semblent pas informatives (même taux >50k que le mode) → imputation par le mode (US). Ses
modalités `<50` obs sont regroupées (`min_frequency = RARE_CATEGORY_MIN_FREQUENCY`)
dans un bac `infrequent_sklearn`, puis one-hot.

In [ ]:
country_cols = [c for c in feat_names if c.startswith("cat_country__")]
n_raw = X_train["native-country"].nunique(dropna=True)
print(f"native-country : {n_raw} modalites brutes  ->  {len(country_cols)} colonnes one-hot")
print("Bac des modalites rares present :", any("infrequent" in c for c in country_cols))

## 5. Vérification de la transformation du jeu de test

Le test est transformé avec `transform` : on
réutilise les statistiques apprises sur le train (pour les features, la cible a déjà été transformée plus tôt) (médianes, moyennes/écarts,
modalités). On n'a par conséquent pas de fuite, et le test produit exactement les
mêmes colonnes.

Etant donnée la structure de notre projet, on ne se sert pas directement des variables déclarées ici, mais on montre simplement que la transformation s'est déroulée correctement

In [ ]:
X_test_t = pre.transform(X_test)

print(f"Train transforme : {X_train_t.shape}")
print(f"Test  transforme : {X_test_t.shape}")
print("Memes colonnes           :", X_test_t.shape[1] == X_train_t.shape[1])
print("NaN dans le test         :", int(X_test_t.isna().sum().sum()))


## 6. Bilan preprocessing

Comme expliqué précédemment, le preprocessing présenté ici est celui par défaut, et on définira la pipeline finale à l'issue de tests d'ablation (vérifier les résultats de certains modèles en utilisant certaines transformations de la pipeline ou au contraire en ne les utilisant pas) dans la partie modélisation.